In [6]:
import pandas as pd
transaction_data= pd.read_csv("C:/Users/lavanya/Downloads/QVI_transaction_data.csv")
customer_data=pd.read_csv("C:/Users/lavanya/Downloads/QVI_purchase_behaviour.csv")

print(customer_data)


       LYLTY_CARD_NBR               LIFESTAGE PREMIUM_CUSTOMER
0                1000   YOUNG SINGLES/COUPLES          Premium
1                1002   YOUNG SINGLES/COUPLES       Mainstream
2                1003          YOUNG FAMILIES           Budget
3                1004   OLDER SINGLES/COUPLES       Mainstream
4                1005  MIDAGE SINGLES/COUPLES       Mainstream
...               ...                     ...              ...
72632         2370651  MIDAGE SINGLES/COUPLES       Mainstream
72633         2370701          YOUNG FAMILIES       Mainstream
72634         2370751          YOUNG FAMILIES          Premium
72635         2370961          OLDER FAMILIES           Budget
72636         2373711   YOUNG SINGLES/COUPLES       Mainstream

[72637 rows x 3 columns]


In [9]:
transaction_data['DATE']=pd.to_datetime(transaction_data['DATE'],unit='D',origin='1899-12-30')
print(transaction_data['DATE'].head())

0   2018-10-17
1   2019-05-14
2   2019-05-20
3   2018-08-17
4   2018-08-18
Name: DATE, dtype: datetime64[ns]


In [10]:
unique_products=transaction_data['PROD_NAME'].unique()
for p in sorted(unique_products):
    print(p)

Burger Rings 220g
CCs Nacho Cheese    175g
CCs Original 175g
CCs Tasty Cheese    175g
Cheetos Chs & Bacon Balls 190g
Cheetos Puffs 165g
Cheezels Cheese 330g
Cheezels Cheese Box 125g
Cobs Popd Sea Salt  Chips 110g
Cobs Popd Sour Crm  &Chives Chips 110g
Cobs Popd Swt/Chlli &Sr/Cream Chips 110g
Dorito Corn Chp     Supreme 380g
Doritos Cheese      Supreme 330g
Doritos Corn Chip Mexican Jalapeno 150g
Doritos Corn Chip Southern Chicken 150g
Doritos Corn Chips  Cheese Supreme 170g
Doritos Corn Chips  Nacho Cheese 170g
Doritos Corn Chips  Original 170g
Doritos Mexicana    170g
Doritos Salsa       Medium 300g
Doritos Salsa Mild  300g
French Fries Potato Chips 175g
Grain Waves         Sweet Chilli 210g
Grain Waves Sour    Cream&Chives 210G
GrnWves Plus Btroot & Chilli Jam 180g
Infuzions BBQ Rib   Prawn Crackers 110g
Infuzions Mango     Chutny Papadums 70g
Infuzions SourCream&Herbs Veg Strws 110g
Infuzions Thai SweetChili PotatoMix 110g
Infzns Crn Crnchers Tangy Gcamole 110g
Kettle 135g Swt Pot S

In [11]:
transaction_data=transaction_data[~transaction_data['PROD_NAME'].str.contains('salsa',case=False)]
print(transaction_data.shape)

(246742, 8)


In [14]:
print(transaction_data['PROD_QTY'].describe())

count    246742.000000
mean          1.908062
std           0.659831
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max         200.000000
Name: PROD_QTY, dtype: float64


In [15]:
outliers=transaction_data[transaction_data['PROD_QTY']==200]
print(outliers)

            DATE  STORE_NBR  LYLTY_CARD_NBR  TXN_ID  PROD_NBR  \
69762 2018-08-19        226          226000  226201         4   
69763 2019-05-20        226          226000  226210         4   

                              PROD_NAME  PROD_QTY  TOT_SALES  
69762  Dorito Corn Chp     Supreme 380g       200      650.0  
69763  Dorito Corn Chp     Supreme 380g       200      650.0  


In [19]:
transaction_data=transaction_data[transaction_data['LYLTY_CARD_NBR'] != 226000]
print(transaction_data.shape)
print(transaction_data['PROD_QTY'].describe())

(246740, 8)
count    246740.000000
mean          1.906456
std           0.342499
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max           5.000000
Name: PROD_QTY, dtype: float64


In [22]:
transaction_data['PACK_SIZE']= transaction_data['PROD_NAME'].str.extract(r'(\d+)(?=g)').astype(float)
print(transaction_data['PACK_SIZE'].describe())

count    240676.000000
mean        175.302286
std          60.014468
min          70.000000
25%         150.000000
50%         170.000000
75%         175.000000
max         380.000000
Name: PACK_SIZE, dtype: float64


In [25]:
transaction_data['BRAND_NAME']=transaction_data['PROD_NAME'].str.extract(r'^(\S+)')[0].str.upper()
print(sorted (transaction_data['BRAND_NAME'].unique()))

['BURGER', 'CCS', 'CHEETOS', 'CHEEZELS', 'COBS', 'DORITO', 'DORITOS', 'FRENCH', 'GRAIN', 'GRNWVES', 'INFUZIONS', 'INFZNS', 'KETTLE', 'NATURAL', 'NCC', 'PRINGLES', 'RED', 'RRD', 'SMITH', 'SMITHS', 'SNBTS', 'SUNBITES', 'THINS', 'TOSTITOS', 'TWISTIES', 'TYRRELLS', 'WOOLWORTHS', 'WW']


In [26]:
brand_fixes = {
    'DORITO': 'DORITOS',
    'RED': 'RRD',
    'SMITH': 'SMITHS',
    'SNBTS': 'SUNBITES',
    'INFZNS': 'INFUZIONS',
    'GRAIN': 'GRNWVES',
    'NCC': 'NATURAL',
    'WW': 'WOOLWORTHS'
}
transaction_data['BRAND_NAME'] = transaction_data['BRAND_NAME'].replace(brand_fixes)
print(sorted(transaction_data['BRAND_NAME'].unique()))

['BURGER', 'CCS', 'CHEETOS', 'CHEEZELS', 'COBS', 'DORITOS', 'FRENCH', 'GRNWVES', 'INFUZIONS', 'KETTLE', 'NATURAL', 'PRINGLES', 'RRD', 'SMITHS', 'SUNBITES', 'THINS', 'TOSTITOS', 'TWISTIES', 'TYRRELLS', 'WOOLWORTHS']


In [28]:

data=transaction_data.merge(customer_data, on='LYLTY_CARD_NBR',how='left')
print(data.shape)
print(data.isnull().sum())

(246740, 12)
DATE                   0
STORE_NBR              0
LYLTY_CARD_NBR         0
TXN_ID                 0
PROD_NBR               0
PROD_NAME              0
PROD_QTY               0
TOT_SALES              0
PACK_SIZE           6064
BRAND_NAME             0
LIFESTAGE              0
PREMIUM_CUSTOMER       0
dtype: int64


In [32]:
print(data[data['PACK_SIZE'].isnull()]['PROD_NAME'].unique())


['Grain Waves Sour    Cream&Chives 210G'
 'Red Rock Deli Sp    Salt & Truffle 150G'
 'Smiths Thinly       Swt Chli&S/Cream175G']


In [33]:
transaction_data['PACK_SIZE'] = transaction_data['PROD_NAME'].str.extract(r'(\d+)(?=[gG])').astype(float)

# Re-run the merge with the corrected data
data = transaction_data.merge(customer_data, on='LYLTY_CARD_NBR', how='left')
print(data['PACK_SIZE'].isnull().sum())

0


In [34]:
sales_summary = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['TOT_SALES'].sum().reset_index()
sales_summary = sales_summary.sort_values('TOT_SALES', ascending=False)
print(sales_summary)

                 LIFESTAGE PREMIUM_CUSTOMER  TOT_SALES
6           OLDER FAMILIES           Budget  156863.75
19   YOUNG SINGLES/COUPLES       Mainstream  147582.20
13                RETIREES       Mainstream  145168.95
15          YOUNG FAMILIES           Budget  129717.95
9    OLDER SINGLES/COUPLES           Budget  127833.60
10   OLDER SINGLES/COUPLES       Mainstream  124648.50
11   OLDER SINGLES/COUPLES          Premium  123537.55
12                RETIREES           Budget  105916.30
7           OLDER FAMILIES       Mainstream   96413.55
14                RETIREES          Premium   91296.65
16          YOUNG FAMILIES       Mainstream   86338.25
1   MIDAGE SINGLES/COUPLES       Mainstream   84734.25
17          YOUNG FAMILIES          Premium   78571.70
8           OLDER FAMILIES          Premium   75242.60
18   YOUNG SINGLES/COUPLES           Budget   57122.10
2   MIDAGE SINGLES/COUPLES          Premium   54443.85
20   YOUNG SINGLES/COUPLES          Premium   39052.30
0   MIDAGE

In [37]:
sales_summary = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['TOT_SALES'].sum().reset_index()
sales_summary = sales_summary.sort_values('TOT_SALES', ascending=False)
print(sales_summary)
customer_summary = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER'])['LYLTY_CARD_NBR'].nunique().reset_index(name='CUSTOMERS')
print(customer_summary.sort_values('CUSTOMERS', ascending=False))
avg_spend = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER']).apply(
    lambda x: x['TOT_SALES'].sum() / x['LYLTY_CARD_NBR'].nunique()
).reset_index(name='AVG_SPEND_PER_CUSTOMER')
print(avg_spend.sort_values('AVG_SPEND_PER_CUSTOMER', ascending=False))

                 LIFESTAGE PREMIUM_CUSTOMER  TOT_SALES
6           OLDER FAMILIES           Budget  156863.75
19   YOUNG SINGLES/COUPLES       Mainstream  147582.20
13                RETIREES       Mainstream  145168.95
15          YOUNG FAMILIES           Budget  129717.95
9    OLDER SINGLES/COUPLES           Budget  127833.60
10   OLDER SINGLES/COUPLES       Mainstream  124648.50
11   OLDER SINGLES/COUPLES          Premium  123537.55
12                RETIREES           Budget  105916.30
7           OLDER FAMILIES       Mainstream   96413.55
14                RETIREES          Premium   91296.65
16          YOUNG FAMILIES       Mainstream   86338.25
1   MIDAGE SINGLES/COUPLES       Mainstream   84734.25
17          YOUNG FAMILIES          Premium   78571.70
8           OLDER FAMILIES          Premium   75242.60
18   YOUNG SINGLES/COUPLES           Budget   57122.10
2   MIDAGE SINGLES/COUPLES          Premium   54443.85
20   YOUNG SINGLES/COUPLES          Premium   39052.30
0   MIDAGE

C:\Users\lavanya\AppData\Local\Temp\ipykernel_1316\1909808970.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_spend = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER']).apply(


In [38]:
avg_pack=data.groupby(['LIFESTAGE','PREMIUM_CUSTOMER'])['PACK_SIZE'].mean().reset_index()
print(avg_pack.sort_values('PACK_SIZE',ascending=False))

                 LIFESTAGE PREMIUM_CUSTOMER   PACK_SIZE
19   YOUNG SINGLES/COUPLES       Mainstream  178.344249
1   MIDAGE SINGLES/COUPLES       Mainstream  177.898693
11   OLDER SINGLES/COUPLES          Premium  176.485568
12                RETIREES           Budget  176.395641
14                RETIREES          Premium  176.368421
4             NEW FAMILIES       Mainstream  175.629748
6           OLDER FAMILIES           Budget  175.546342
15          YOUNG FAMILIES           Budget  175.459720
9    OLDER SINGLES/COUPLES           Budget  175.334673
5             NEW FAMILIES          Premium  175.245296
13                RETIREES       Mainstream  175.213671
7           OLDER FAMILIES       Mainstream  175.175666
10   OLDER SINGLES/COUPLES       Mainstream  174.812145
3             NEW FAMILIES           Budget  174.766643
17          YOUNG FAMILIES          Premium  174.659032
2   MIDAGE SINGLES/COUPLES          Premium  174.585391
8           OLDER FAMILIES          Premium  174